# S7.1 — Conversation Memory

Session-based conversation history backed by Redis with:
- Sliding window (last 20 messages, configurable)
- 24h TTL per session, refreshed on every interaction
- Graceful degradation when Redis is unavailable
- `ChatMessage` Pydantic model with role, content, timestamp, metadata

In [1]:
import sys
sys.path.insert(0, "../..")

from src.services.chat.memory import ChatMessage, ConversationMemory, make_conversation_memory, VALID_ROLES, KEY_PREFIX

print(f"ChatMessage:          {ChatMessage}")
print(f"ConversationMemory:   {ConversationMemory}")
print(f"make_conversation_memory: {make_conversation_memory}")
print(f"VALID_ROLES:          {VALID_ROLES}")
print(f"KEY_PREFIX:           {KEY_PREFIX}")
print("\n✓ All imports successful")

ChatMessage:          <class 'src.services.chat.memory.ChatMessage'>
ConversationMemory:   <class 'src.services.chat.memory.ConversationMemory'>
make_conversation_memory: <function make_conversation_memory at 0x105441620>
VALID_ROLES:          ('user', 'assistant')
KEY_PREFIX:           chat:session:

✓ All imports successful


## FR-7: ChatMessage Model

Pydantic model with role validation (user/assistant only), auto-timestamp, and optional metadata.

In [2]:
from datetime import UTC, datetime

# Valid message creation
msg = ChatMessage(role="user", content="What is attention in transformers?")
print(f"Role:      {msg.role}")
print(f"Content:   {msg.content}")
print(f"Timestamp: {msg.timestamp}")
print(f"Metadata:  {msg.metadata}")

# Auto-timestamp
assert msg.timestamp is not None
assert isinstance(msg.timestamp, datetime)

# With metadata
msg2 = ChatMessage(
    role="assistant",
    content="Attention is a mechanism...",
    metadata={"sources": [{"title": "Attention Is All You Need", "arxiv_id": "1706.03762"}]}
)
assert msg2.metadata is not None
print(f"\nWith metadata: {msg2.metadata}")

# Serialization roundtrip
json_str = msg.model_dump_json()
restored = ChatMessage.model_validate_json(json_str)
assert restored.role == msg.role
assert restored.content == msg.content
print(f"\nJSON roundtrip: ✓")

# Invalid role rejected
try:
    ChatMessage(role="system", content="should fail")
    assert False, "Should have raised"
except ValueError as e:
    print(f"Invalid role rejected: ✓ ({e})")

print("\n✓ ChatMessage model works correctly")

Role:      user
Content:   What is attention in transformers?
Timestamp: 2026-03-11 21:20:46.216932+00:00
Metadata:  None

With metadata: {'sources': [{'title': 'Attention Is All You Need', 'arxiv_id': '1706.03762'}]}

JSON roundtrip: ✓
Invalid role rejected: ✓ (1 validation error for ChatMessage
role
  Input should be 'user' or 'assistant' [type=literal_error, input_value='system', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/literal_error)

✓ ChatMessage model works correctly


## FR-1 to FR-6: ConversationMemory with Mocked Redis

Test add_message, get_history, sliding window, clear_session, list_sessions, TTL refresh — all using a mock Redis client.

In [3]:
import json
from unittest.mock import AsyncMock, MagicMock

# --- Simulate a Redis-backed conversation ---
stored_messages: list[str] = []

mock_redis = AsyncMock()

async def fake_rpush(key, value):
    stored_messages.append(value)
    return len(stored_messages)

async def fake_lrange(key, start, end):
    if end == -1:
        return stored_messages[start:] if start >= 0 else stored_messages[start:]
    return stored_messages[start:end+1]

async def fake_llen(key):
    return len(stored_messages)

mock_redis.rpush = AsyncMock(side_effect=fake_rpush)
mock_redis.lrange = AsyncMock(side_effect=fake_lrange)
mock_redis.llen = AsyncMock(side_effect=fake_llen)
mock_redis.ltrim = AsyncMock(return_value=True)
mock_redis.expire = AsyncMock(return_value=True)
mock_redis.delete = AsyncMock(return_value=1)

memory = ConversationMemory(redis_client=mock_redis, max_messages=5, ttl_seconds=86400)

# Add messages
await memory.add_message("demo-session", "user", "What is BERT?")
await memory.add_message("demo-session", "assistant", "BERT is a transformer model...",
                          metadata={"sources": [{"title": "BERT paper"}]})
await memory.add_message("demo-session", "user", "How does it compare to GPT?")

# Retrieve history
history = await memory.get_history("demo-session")
print(f"Messages in session: {len(history)}")
for msg in history:
    print(f"  [{msg.role}] {msg.content[:50]}...")

# TTL refresh called
assert mock_redis.expire.called
print(f"\nTTL refreshed: ✓")

# Validation: empty content rejected
try:
    await memory.add_message("demo-session", "user", "")
except ValueError:
    print("Empty content rejected: ✓")

# Validation: invalid role rejected
try:
    await memory.add_message("demo-session", "system", "hello")
except ValueError:
    print("Invalid role rejected: ✓")

# Clear session
result = await memory.clear_session("demo-session")
assert result is True
print(f"Session cleared: ✓")

# Graceful Redis failure
mock_redis.rpush.side_effect = ConnectionError("Redis down")
await memory.add_message("sess-fail", "user", "this should not raise")
print("Redis failure handled gracefully: ✓")

print("\n✓ All ConversationMemory operations verified")

Failed to add message to session sess-fail (graceful skip)
Traceback (most recent call last):
  File "/Users/nishantgaurav/Project/PaperAlchemy/notebooks/specs/../../src/services/chat/memory.py", line 71, in add_message
    await self._redis.rpush(key, message.model_dump_json())
  File "/opt/anaconda3/lib/python3.12/unittest/mock.py", line 2282, in _execute_mock_call
    raise effect
ConnectionError: Redis down


Messages in session: 3
  [user] What is BERT?...
  [assistant] BERT is a transformer model......
  [user] How does it compare to GPT?...

TTL refreshed: ✓
Empty content rejected: ✓
Invalid role rejected: ✓
Session cleared: ✓
Redis failure handled gracefully: ✓

✓ All ConversationMemory operations verified


## FR-8: Factory + Dependency Injection

Verify `make_conversation_memory()` factory and `ConversationMemoryDep` are available.

In [4]:
from src.dependency import (
    ConversationMemoryDep,
    get_conversation_memory,
    set_conversation_memory,
)

# Verify DI functions exist
assert callable(get_conversation_memory)
assert callable(set_conversation_memory)
print(f"ConversationMemoryDep: {ConversationMemoryDep}")
print(f"get_conversation_memory: {get_conversation_memory}")
print(f"set_conversation_memory: {set_conversation_memory}")

# Test setter/getter
set_conversation_memory(None)
assert get_conversation_memory() is None

mock_mem = ConversationMemory(redis_client=AsyncMock(), max_messages=20, ttl_seconds=86400)
set_conversation_memory(mock_mem)
assert get_conversation_memory() is mock_mem
set_conversation_memory(None)  # cleanup

print("\n✓ DI wiring verified")

ConversationMemoryDep: typing.Annotated[src.services.chat.memory.ConversationMemory | None, Depends(dependency=<function get_conversation_memory at 0x17af10ae0>, use_cache=True, scope=None)]
get_conversation_memory: <function get_conversation_memory at 0x17af10ae0>
set_conversation_memory: <function set_conversation_memory at 0x17af10a40>

✓ DI wiring verified


## Test Suite Verification

In [5]:
import subprocess
result = subprocess.run(
    ["python", "-m", "pytest", "tests/unit/test_conversation_memory.py", "-v", "--tb=short"],
    capture_output=True, text=True, cwd="../.."
)
print(result.stdout[-1500:] if len(result.stdout) > 1500 else result.stdout)
if result.returncode == 0:
    print("\n✓ All 29 tests passing")
else:
    print(f"\n✗ Tests failed (exit code {result.returncode})")
    print(result.stderr[-500:])

t_empty_session PASSED [ 58%]
tests/unit/test_conversation_memory.py::TestGetHistory::test_with_limit PASSED [ 62%]
tests/unit/test_conversation_memory.py::TestGetHistory::test_redis_error_returns_empty PASSED [ 65%]
tests/unit/test_conversation_memory.py::TestGetHistory::test_refreshes_ttl PASSED [ 68%]
tests/unit/test_conversation_memory.py::TestGetHistory::test_corrupted_data_skipped PASSED [ 72%]
tests/unit/test_conversation_memory.py::TestClearSession::test_success PASSED [ 75%]
tests/unit/test_conversation_memory.py::TestClearSession::test_not_found PASSED [ 79%]
tests/unit/test_conversation_memory.py::TestClearSession::test_redis_error PASSED [ 82%]
tests/unit/test_conversation_memory.py::TestListSessions::test_success PASSED [ 86%]
tests/unit/test_conversation_memory.py::TestListSessions::test_redis_error PASSED [ 89%]
tests/unit/test_conversation_memory.py::TestFactory::test_make_conversation_memory_success PASSED [ 93%]
tests/unit/test_conversation_memory.py::TestFactory::tes